In [ ]:
!pip install openai PyPDF2 pandas -q

import os
import re
import gradio as gr
import smtplib
import pandas as pd
from dataclasses import dataclass, asdict
from email.message import EmailMessage
from PyPDF2 import PdfReader
from typing import Dict, List, Any, Tuple
from google.colab import userdata
from openai import OpenAI
import time

# ============================================================
# CONFIGURATION — Modified for dynamic uploads
# ============================================================

INVOICE_DIR = "invoices"
REPORT_DIR = "reports"
INTERMEDIATE_DIR = "intermediate"

os.makedirs(INVOICE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

OPENAI_API_KEY = userdata.get("KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

LLM_MODEL = "gpt-4o"


# ============================================================
# UTILITIES (unchanged)
# ============================================================

def clean_part_name(name: str) -> str:
    if not isinstance(name, str):
        name = str(name)
    name = " ".join(name.split())
    name = re.sub(r"\s*-\s*", "-", name)
    return name.strip()


def call_llm(system_prompt: str, user_prompt: str, model: str = LLM_MODEL) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()


# ============================================================
# (Your AGENT CLASSES remain fully unchanged)
# ============================================================

@dataclass
class ErrorManualEntry:
    error_code: str
    description: str
    possible_causes: List[str]
    recommended_parts: List[str]
    severity: str


@dataclass
class Agent1OutputRow:
    timestamp: str
    machine_id: str
    machine_name: str
    error_code: str
    severity: str
    part_name: str


@dataclass
class ErrorPartVendorRow:
    timestamp: str
    machine_id: str
    machine_name: str
    error_code: str
    severity: str
    part_name: str
    vendor: str
    price: float
    delivery_time: str


@dataclass
class InvoiceLine:
    part_name: str
    quantity: int
    unit_price: float
    delivery_time: str
    subtotal: float

# ============================================================
# AGENT 1: LOG ANALYSER
# ============================================================

class LogAnalyzerAgent:
    """
    Agent 1:
    - Parse Error_Reference_Manual.pdf into structured entries:
        Error Code, Description, Possible Causes, Recommended Parts, Severity
    - Compare with PLT4_100_logs.csv:
        For each error in logs, find severity + recommended parts.
    - Explode to one row per (error, part) and save to intermediate/agent1_output.csv
      (this CSV is then used by Agent 2).
    """

    def __init__(self, manual_path: str, log_path: str, output_dir: str = "intermediate"):
        self.manual_path = manual_path
        self.log_path = log_path
        self.output_dir = output_dir
        self.error_manual: Dict[str, ErrorManualEntry] = {}

    # ------------ NEW: manual parser tuned to your reference manual ------------
    def _parse_error_manual(self) -> Dict[str, ErrorManualEntry]:
        """
        Parses a manual of the form:

        Error Code: E108
        1) Error Description: ...
        2) Possible Causes:
        a) ...
        b) ...
        3) Recommended Part Replacement:
        a) ...
        b) ...
        4) Severity: Medium / High / Low
        """
        reader = PdfReader(self.manual_path)
        full_text = "".join(page.extract_text() or "" for page in reader.pages)

        # Normalize whitespace to make regex matching more robust
        text = " ".join(full_text.split())

        # Split into blocks, each starting with "Error Code: <E###>"
        # Example: "Error Code: E108 1) Error Description: ..."
        blocks = re.split(r"Error Code:\s*(E\d+)", text)
        # re.split gives: ["prefix", "E108", " rest of block ", "E203", " rest of block ", ...]
        # We'll pair them as (code, block_text)
        manual: Dict[str, ErrorManualEntry] = {}

        for i in range(1, len(blocks), 2):
            error_code = blocks[i]
            block_text = blocks[i + 1]

            # 1) Description
            desc_match = re.search(
                r"1\)\s*Error Description:\s*(.*?)(?:2\)\s*Possible Causes:)",
                block_text,
                re.IGNORECASE,
            )
            description = desc_match.group(1).strip() if desc_match else ""

            # 2) Possible Causes: grab text between "2) Possible Causes:" and "3) Recommended Part Replacement:"
            causes_match = re.search(
                r"2\)\s*Possible Causes:\s*(.*?)(?:3\)\s*Recommended Part Replacement:)",
                block_text,
                re.IGNORECASE,
            )
            causes_raw = causes_match.group(1).strip() if causes_match else ""
            # Split causes on "a) ... b) ... c) ..." pattern
            causes = [
                c.strip(" .")
                for c in re.split(r"[a-z]\)", causes_raw)
                if c.strip()
            ]

            # 3) Recommended Parts: between "3) Recommended Part Replacement:" and "4) Severity:"
            parts_match = re.search(
                r"3\)\s*Recommended Part Replacement:\s*(.*?)(?:4\)\s*Severity:)",
                block_text,
                re.IGNORECASE,
            )
            parts_raw = parts_match.group(1).strip() if parts_match else ""
            recommended_parts = [
              clean_part_name(p.strip(" ."))
              for p in re.split(r"[a-z]\)", parts_raw)
              if p.strip()
          ]

            # 4) Severity: directly match Low / Medium / High (or any word after "Severity:")
            severity_match = re.search(
                r"4\)\s*Severity:\s*([A-Za-z]+)",
                block_text,
                re.IGNORECASE,
            )
            severity = severity_match.group(1).capitalize() if severity_match else "Unknown"

            manual[error_code] = ErrorManualEntry(
                error_code=error_code,
                description=description,
                possible_causes=causes,
                recommended_parts=recommended_parts,
                severity=severity,
            )

        print(f"[Agent 1] Parsed {len(manual)} error codes from manual.")
        return manual


    def _load_logs(self) -> pd.DataFrame:
        df = pd.read_csv(self.log_path)
        df["error_code"] = df["error_code"].astype(str).str.strip()
        df = df[(df["error_code"].notna()) & (df["error_code"] != "None")]
        return df

    def run(self) -> Tuple[pd.DataFrame, str]:
        print("[Agent 1] Parsing error manual...")
        self.error_manual = self._parse_error_manual()

        print("[Agent 1] Loading logs...")
        logs_df = self._load_logs()

        print("[Agent 1] Matching each log error with manual (severity + parts)...")
        output_rows: List[Agent1OutputRow] = []

        for _, row in logs_df.iterrows():
            code = row["error_code"]
            manual_entry = self.error_manual.get(code)

            if manual_entry:
                severity = manual_entry.severity
                parts = manual_entry.recommended_parts
            else:
                # If error code not found in manual, mark as Unknown
                severity = "Unknown"
                parts = []

            if not parts:
                # If no recommended parts, still record row with part_name as "Unknown" if you want
                output_rows.append(
                    Agent1OutputRow(
                        timestamp=str(row["timestamp"]),
                        machine_id=str(row["machine_id"]),
                        machine_name=str(row["machine_name"]),
                        error_code=code,
                        severity=severity,
                        part_name="Unknown",
                    )
                )
            else:
                # One row per recommended part
                for part in parts:
                  output_rows.append(
                      Agent1OutputRow(
                          timestamp=str(row["timestamp"]),
                          machine_id=str(row["machine_id"]),
                          machine_name=str(row["machine_name"]),
                          error_code=code,
                          severity=severity,
                          part_name=clean_part_name(part),
                      )
                  )

        agent1_df = pd.DataFrame([asdict(r) for r in output_rows])

        agent1_output_path = os.path.join(self.output_dir, "agent1_output.csv")
        agent1_df.to_csv(agent1_output_path, index=False)
        print(f"[Agent 1] Completed. Output saved to: {agent1_output_path}")
        print(f"[Agent 1] Rows: {len(agent1_df)}")

        return agent1_df, agent1_output_path


# ============================================================
# AGENT 2: VENDOR PURCHASE ORDER CREATION
# ============================================================

class VendorPOAgent:
    """
    - Takes Agent 1's output file (agent1_output.csv)
    - Compares part_name from Agent 1 with vendor_catalog.csv
    - For every matching (part, vendor) combination, creates a record
    - Groups by vendor and generates a full purchase order CSV per vendor
      containing ALL parts that vendor can supply (with cost & delivery time).
    """

    def __init__(self, vendor_catalog_path: str, invoice_dir: str = INVOICE_DIR):
        self.vendor_catalog_path = vendor_catalog_path
        self.invoice_dir = invoice_dir
        self.vendor_df: pd.DataFrame | None = None

    def _load_vendor_catalog(self) -> pd.DataFrame:
      df = pd.read_csv(self.vendor_catalog_path)
      df["part_name"] = df["part_name"].astype(str).apply(clean_part_name)
      df["vendor"] = df["vendor"].astype(str).str.strip()
      return df

    def run(self, agent1_output_path: str) -> Tuple[pd.DataFrame, Dict[str, str]]:
        print("[Agent 2] Loading vendor catalog...")
        self.vendor_df = self._load_vendor_catalog()

        print("[Agent 2] Loading Agent 1 output...")
        agent1_df = pd.read_csv(agent1_output_path)
        agent1_df["part_name"] = agent1_df["part_name"].astype(str).apply(clean_part_name)

        # Join Agent 1 (errors+parts) with vendor catalog on part_name
        print("[Agent 2] Matching all parts from Agent 1 with vendors...")
        merged = pd.merge(
          agent1_df,
          self.vendor_df,
          on="part_name",
          how="left",
          suffixes=("", "_vendor"),
      )

        # Build ErrorPartVendorRow objects only for rows where vendor is present
        error_part_vendor_rows: List[ErrorPartVendorRow] = []
        missing_parts: List[str] = []

        for _, row in merged.iterrows():
            if pd.isna(row.get("vendor")):
                missing_parts.append(row["part_name"])
                continue

            error_part_vendor_rows.append(
                ErrorPartVendorRow(
                    timestamp=str(row["timestamp"]),
                    machine_id=str(row["machine_id"]),
                    machine_name=str(row["machine_name"]),
                    error_code=str(row["error_code"]),
                    severity=str(row["severity"]),
                    part_name=str(row["part_name"]),
                    vendor=str(row["vendor"]),
                    price=float(row["price"]),
                    delivery_time=str(row["delivery_time"]),
                )
            )

        if missing_parts:
            print("[Agent 2] WARNING: Some parts had no matching vendor:")
            print("  ", sorted(set(missing_parts)))

        epv_df = pd.DataFrame([asdict(r) for r in error_part_vendor_rows])
        print(f"[Agent 2] Found {len(epv_df)} error-part-vendor records.")

        # === Create invoices per vendor containing ALL parts that vendor supplies ===
        print("[Agent 2] Creating purchase orders per vendor...")
        invoice_status: Dict[str, str] = {}

        for vendor_name, group in epv_df.groupby("vendor"):
            lines: List[InvoiceLine] = []

            # Group by part_name to compute quantity for that vendor
            part_grouped = group.groupby("part_name")
            for part_name, pg in part_grouped:
                unit_price = float(pg["price"].iloc[0])
                quantity = len(pg)
                subtotal = unit_price * quantity
                delivery_time = str(pg["delivery_time"].iloc[0])

                lines.append(
                    InvoiceLine(
                        part_name=part_name,
                        quantity=quantity,
                        unit_price=unit_price,
                        delivery_time=delivery_time,
                        subtotal=subtotal,
                    )
                )

            invoice_df = pd.DataFrame([asdict(l) for l in lines])
            invoice_df["total_vendor_cost"] = invoice_df["subtotal"].sum()

            filename = os.path.join(self.invoice_dir, f"invoice_{vendor_name}.csv")
            invoice_df.to_csv(filename, index=False)
            invoice_status[vendor_name] = filename

            print(f"  - Purchase order created for {vendor_name}: {filename}")

        return epv_df, invoice_status


# ============================================================
# AGENT 3: VENDOR PURCHASE ORDER EMAILING
# ============================================================

class EmailAgent:
    """
    - Uses Vendor→invoice mapping from Agent 2
    - Sends email with invoice attached to each vendor
    - Returns email_status dict
    """

    def __init__(self,
                 smtp_server: str,
                 smtp_port: int,
                 user_email: str,
                 user_password: str,
                 vendor_emails: Dict[str, str]):
        self.smtp_server = smtp_server
        self.smtp_port = smtp_port
        self.user_email = user_email
        self.user_password = user_password
        self.vendor_emails = vendor_emails

    def _send_email_with_attachment(self, to_email: str, subject: str, body: str, attachment_path: str):
        msg = EmailMessage()
        msg["From"] = self.user_email
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.set_content(body)

        with open(attachment_path, "rb") as f:
            data = f.read()
        msg.add_attachment(
            data,
            maintype="text",
            subtype="csv",
            filename=os.path.basename(attachment_path),
        )

        with smtplib.SMTP(self.smtp_server, self.smtp_port) as server:
            server.starttls()
            server.login(self.user_email, self.user_password)
            server.send_message(msg)

    def run(self, invoice_files: Dict[str, str]) -> Dict[str, str]:
        print("[Agent 3] Sending purchase order emails...")
        email_status: Dict[str, str] = {}

        for vendor, invoice_path in invoice_files.items():
            to_email = self.vendor_emails.get(vendor)
            if not to_email:
                email_status[vendor] = "no_email_configured"
                print(f"  - SKIP {vendor}: No email configured.")
                continue

            subject = f"Purchase Order - {vendor}"
            body = (
                f"Dear {vendor} Team,\n\n"
                "Please find attached the purchase order for required spare parts.\n"
                "Kindly confirm availability and expected delivery schedule.\n\n"
                f"Regards,\n{self.user_email}"
            )

            try:
                self._send_email_with_attachment(to_email, subject, body, invoice_path)
                email_status[vendor] = "sent"
                print(f"  - Email sent to {vendor} ({to_email}).")
            except Exception as e:
                email_status[vendor] = f"failed: {e}"
                print(f"  - FAILED to send email to {vendor}: {e}")

        return email_status


# ============================================================
# AGENT 4: SUMMARY REPORT
# ============================================================

class SummaryReportAgent:
    """
    - Takes:
        * Agent 1 output (errors + parts + severity)
        * Agent 2 output (error-part-vendor mapping)
        * Email status (Agent 3)
    - Produces:
        1) Tabular CSV summary
        2) Separate text summary report (.txt), which uses the tabular summary as input
    """

    def __init__(self, report_dir: str = REPORT_DIR):
        self.report_dir = report_dir

    def run(
        self,
        agent1_df: pd.DataFrame,
        epv_df: pd.DataFrame,
        email_status: Dict[str, str],
    ) -> Tuple[str, str]:
        print("[Agent 4] Preparing tabular summary report...")

        # Merge Agent1 (errors+parts) with Agent2 (vendor info)
        merged = pd.merge(
            agent1_df,
            epv_df,
            on=["timestamp", "machine_id", "machine_name", "error_code", "severity", "part_name"],
            how="left",
            suffixes=("_a1", "_a2"),
        )

        # Attach email status per vendor
        merged["vendor_email_status"] = merged["vendor"].apply(
            lambda v: email_status.get(v, "no_po_generated") if pd.notna(v) else "no_po_generated"
        )

        # Columns for final summary
        report_cols = [
            "timestamp",
            "machine_id",
            "machine_name",
            "error_code",
            "severity",
            "part_name",
            "vendor",
            "price",
            "delivery_time",
            "vendor_email_status",
        ]
        summary_df = merged[report_cols].sort_values(
            ["timestamp", "machine_id", "error_code", "part_name"]
        )

        # 1) TABULAR SUMMARY CSV
        summary_csv_path = os.path.join(self.report_dir, "tabular_summary_report.csv")
        summary_df.to_csv(summary_csv_path, index=False)
        print(f"[Agent 4] Tabular summary saved to: {summary_csv_path}")

        # 2) TEXT SUMMARY REPORT (.txt) USING TABULAR SUMMARY AS INPUT
        print("[Agent 4] Generating text-based summary report using the tabular summary...")
        # Some basic stats derived from the tabular summary:
        severity_counts = summary_df.groupby("severity")["error_code"].count().to_dict()
        vendor_costs = summary_df.dropna(subset=["price"]).groupby("vendor")["price"].sum().to_dict()
        email_status_counts = summary_df.groupby("vendor_email_status")["error_code"].count().to_dict()

        severity_str = ", ".join([f"{sev}: {cnt} occurrences" for sev, cnt in severity_counts.items()])
        vendor_cost_str = ", ".join([f"{v}: {amt:.2f}" for v, amt in vendor_costs.items()])
        email_status_str = ", ".join([f"{st}: {cnt}" for st, cnt in email_status_counts.items()])

        # We pass a concise representation of the tabular summary to the LLM as context
        text_summary = call_llm(
            system_prompt=(
                """You are an expert manufacturing analyst and corporate reporting specialist.
                You generate clear, structured, highly professional reports for plant leadership.
                Your writing style must be:

                - detailed

                - insight-driven

                - business-oriented

                - readable at senior-management level

                - action-oriented, highlighting operational risks & next steps

                - Ensure the report is formatted with proper headings, bullet points, tables, and executive summaries."""
            ),
            user_prompt=(
                "Here is the summary table data in aggregated form:\n\n"
                f"- Errors by severity: {severity_str}\n"
                f"- Total parts cost by vendor in Indian Rupees: {vendor_cost_str}\n"
                f"- Email / PO status counts: {email_status_str}\n\n"
                "Generate a comprehensive,detailed, plant-head–ready operational report. "
                "Highlight most critical issues, which machines / errors drive cost, "
                "and whether purchase orders have been successfully initiated."
                "Output format:"
                "- Return the report without any bold or Heading type special characters like * or #"
                f"-Ignore the Unknown errors from {severity_str} which are actually entires without any errors"
                f"-Ignore the instances where no PO's were generated as they are actually entires without any errors"
            ),

        )

        text_summary_path = os.path.join(self.report_dir, "text_summary_report.txt")
        with open(text_summary_path, "w", encoding="utf-8") as f:
            f.write(text_summary)

        print(f"[Agent 4] Text summary report saved to: {text_summary_path}")

        return summary_csv_path, text_summary_path

def send_report_email(to_email: str, attachment_path: str):
    """
    Send the text summary report to the plant head as an email attachment.
    Uses the same SMTP creds entered in Section 1 (GLOBAL_STATE['email'], 'password').
    """
    msg = EmailMessage()
    msg["From"] = GLOBAL_STATE["email"]
    msg["To"] = to_email
    msg["Subject"] = "Plant Maintenance Summary Report"

    msg.set_content(
        "Dear Sir/Madam,\n\n"
        "Please find attached the latest maintenance summary report for the plant.\n\n"
        "Regards,\n"
        "Maintenance Automation System"
    )

    with open(attachment_path, "rb") as f:
        data = f.read()

    # Attach as plain text file
    msg.add_attachment(
        data,
        maintype="text",
        subtype="plain",
        filename=os.path.basename(attachment_path),
    )

    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()
        server.login(GLOBAL_STATE["email"], GLOBAL_STATE["password"])
        server.send_message(msg)



# ============================================================
# WRAPPER STATE — Store intermediate files between button clicks
# ============================================================

GLOBAL_STATE = {
    "machine_logs_path": None,
    "error_manual_path": None,
    "vendor_catalog_path": None,
    "agent1_df": None,
    "agent1_output_path": None,
    "epv_df": None,
    "invoice_files": None,
    "email_status": None,
}


# ============================================================
# WRAPPER FUNCTIONS FOR GRADIO
# ============================================================

def upload_inputs(machine_logs, error_manual, vendor_catalog, email, password):
    GLOBAL_STATE["machine_logs_path"] = machine_logs.name
    GLOBAL_STATE["error_manual_path"] = error_manual.name
    GLOBAL_STATE["vendor_catalog_path"] = vendor_catalog.name
    GLOBAL_STATE["email"] = email
    GLOBAL_STATE["password"] = password
    return "Uploads successful. Personal details saved."


def run_agent1():
    agent1 = LogAnalyzerAgent(
        GLOBAL_STATE["error_manual_path"],
        GLOBAL_STATE["machine_logs_path"],
        INTERMEDIATE_DIR
    )
    df, path = agent1.run()

    GLOBAL_STATE["agent1_df"] = df
    GLOBAL_STATE["agent1_output_path"] = path
    return f"Agent 1 completed.\nRows processed: {len(df)}\nSaved at: {path}"


def run_agent2():
    agent2 = VendorPOAgent(GLOBAL_STATE["vendor_catalog_path"], INVOICE_DIR)
    epv_df, invoice_files = agent2.run(GLOBAL_STATE["agent1_output_path"])

    GLOBAL_STATE["epv_df"] = epv_df
    GLOBAL_STATE["invoice_files"] = invoice_files
    return f"Agent 2 completed.\nInvoices generated: {invoice_files}"


def run_agent3(v1, v2, v3, v4, v5):
    # Get the vendors for which invoices were actually generated
    invoice_files = GLOBAL_STATE["invoice_files"]
    invoice_vendors = list(invoice_files.keys())  # e.g. ["3M", "Bosch", "Crompton", "Danfoss", "Eaton", ...]

    # Map the first 5 emails entered to the first 5 vendors in the invoice list
    input_emails = [v1, v2, v3, v4, v5]
    vendor_emails: Dict[str, str] = {}

    for idx, email in enumerate(input_emails):
        if email and idx < len(invoice_vendors):
            vendor_name = invoice_vendors[idx]
            vendor_emails[vendor_name] = email

    # Create EmailAgent with this mapping
    agent3 = EmailAgent(
        smtp_server=SMTP_SERVER,
        smtp_port=SMTP_PORT,
        user_email=GLOBAL_STATE["email"],
        user_password=GLOBAL_STATE["password"],
        vendor_emails=vendor_emails,
    )

    email_status = agent3.run(invoice_files)
    GLOBAL_STATE["email_status"] = email_status

    # Show which vendor got mapped to which email for transparency
    mapping_str = ", ".join([f"{v}: {e}" for v, e in vendor_emails.items()])
    return f"Agent 3 completed.\nVendor-email mapping used: {mapping_str}\nEmail status: {email_status}"



def run_agent4(plant_head_email: str):
    if not plant_head_email:
        return "Error: Please enter the Plant Head's email before running Agent 4."

    if GLOBAL_STATE.get("agent1_df") is None or GLOBAL_STATE.get("epv_df") is None:
        return "Error: Please run Agent 1 and Agent 2 (and Agent 3, if needed) before Agent 4."

    agent4 = SummaryReportAgent(REPORT_DIR)
    csv_path, txt_path = agent4.run(
        GLOBAL_STATE["agent1_df"],
        GLOBAL_STATE["epv_df"],
        GLOBAL_STATE.get("email_status", {}),
    )

    # Try sending the email with the text report attached
    try:
        send_report_email(plant_head_email, txt_path)
        status = (
            f"Summary report generated and emailed to {plant_head_email}.\n"
            f"Tabular CSV: {csv_path}\n"
            f"Text report: {txt_path}"
        )
    except Exception as e:
        status = (
            "Summary report generated, but FAILED to send email.\n"
            f"Error: {e}\n"
            f"Tabular CSV: {csv_path}\n"
            f"Text report: {txt_path}"
        )

    return status

def validate_openai_key() -> Tuple[bool, str]:
    """
    Validates whether the OpenAI API key is present and usable.
    Returns (is_valid, message).
    """
    if not OPENAI_API_KEY:
        return False, "OPENAI_API_KEY not found. Please set it in Colab → Secrets."

    try:
        # Minimal, safe API call to validate key
        client.models.list()
        return True, "OpenAI API key validated successfully."
    except Exception as e:
        return False, f"Invalid or unusable OpenAI API key: {str(e)}"


def run_full_pipeline(
    machine_logs, error_manual, vendor_catalog, email, password,
    v1, v2, v3, v4, v5,
    plant_head_email
):
    log = ""

    def push(line: str):
        nonlocal log
        log += line + "\n"
        return log.rstrip()

    # -----------------------------
    # Validation
    # -----------------------------
    missing = []
    if machine_logs is None: missing.append("Machine Logs CSV")
    if error_manual is None: missing.append("Error Reference Manual PDF")
    if vendor_catalog is None: missing.append("Vendor Catalog CSV")
    if not email: missing.append("Your Email ID")
    if not password: missing.append("Password")
    if not plant_head_email: missing.append("Plant Head Email")

    if missing:
        yield push("Cannot start. Please fill the following before clicking Run Agent:")
        for m in missing:
            yield push(f"- {m}")
        return

    # -----------------------------
    # Step 0: Upload + auth
    # -----------------------------
    yield push("Saving uploads & authentication details...")
    upload_inputs(machine_logs, error_manual, vendor_catalog, email, password)
    yield push("Uploads successful. Personal details saved.")

    # -----------------------------
    # Agent 1
    # -----------------------------
    yield push("\n\nAgent 1 (Log Analyzer): Analyzes machine logs to identify errors, determine their severity, and map each error to the required replacement parts.\nAgent 1 running...")
    is_valid, msg = validate_openai_key()
    if not is_valid:
        yield push("Error: Execution aborted due to invalid OpenAI API key.")
        return
    a1_start = time.time()
    time.sleep(5)
    a1_status = run_agent1()
    a1_end = time.time()
    yield push(f"{a1_status}\n\nAgent 1 completed in {a1_end - a1_start:.2f} seconds")

    # -----------------------------
    # Agent 2
    # -----------------------------
    yield push("\n\nAgent 2 (Purchase Order Generator): Matches identified parts with vendors list and automatically generates vendor-wise purchase orders with cost.\nAgent 2 running...")
    a2_start = time.time()
    time.sleep(10)
    a2_status = run_agent2()
    a2_end = time.time()
    yield push(f"{a2_status}\n\nAgent 2 completed in {a2_end - a2_start:.2f} seconds")

    # -----------------------------
    # Agent 3
    # -----------------------------
    yield push("\n\nAgent 3 (Vendor Communication): Sends the generated purchase orders to the respective vendors via email.\nAgent 3 running...")
    a3_start = time.time()
    a3_status = run_agent3(v1, v2, v3, v4, v5)
    a3_end = time.time()
    yield push(f"{a3_status}\n\nAgent 3 completed in {a3_end - a3_start:.2f} seconds")

    # -----------------------------
    # Agent 4
    # -----------------------------
    yield push("\n\nAgent 4 (Summary & Reporting): Generates a final summary report based on the output of previous agents and emails it to the Plant Head.\nAgent 4 running...")
    a4_start = time.time()
    a4_status = run_agent4(plant_head_email)
    a4_end = time.time()
    yield push(f"{a4_status}\n\nAgent 4 completed in {a4_end - a4_start:.2f} seconds")

    # -----------------------------
    # Final summary
    # -----------------------------
    total_time = (a4_end - a1_start)
    yield push(f"\n\nAgentic Workflow Execution complete.\nTotal execution time: {total_time:.2f} seconds")





# ============================================================
# GRADIO UI - ENHANCED VERSION
# ============================================================

# Import CSS for custom styling
import gradio as gr

# Custom CSS for better UI
custom_css = """
/* Main styling */
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}
.important-note {
    background: #e8f4fd;
    padding: 12px;
    border-radius: 8px;
    border-left: 4px solid #2196F3;
    margin: 10px 0;
    color:black;
}
.success-box {
    background: #e8f5e9;
    padding: 12px;
    border-radius: 8px;
    border-left: 4px solid #4CAF50;
}
.warning-box {
    background: #fff8e1;
    padding: 12px;
    border-radius: 8px;
    border-left: 4px solid #ff9800;
}
.section-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
    padding: 15px 20px;
    border-radius: 8px;
    margin: 20px 0 15px 0;
    text-align: center;
    font-size: 1.5em;
    font-weight: bold;
}
.card {
    background: white;
    padding: 20px;
    border-radius: 10px;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
    margin: 15px 0;
}
.button-primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    border: none !important;
    font-weight: 600 !important;
}
.button-secondary {
    background: #6c757d !important;
    color: white !important;
    border: none !important;
}
.button-success {
    background: #28a745 !important;
    color: white !important;
    border: none !important;
}
.step-number {
    display: inline-block;
    width: 28px;
    height: 28px;
    background: #2196F3;
    color: white;
    text-align: center;
    border-radius: 50%;
    margin-right: 10px;
    font-weight: bold;
}
.center-row {
    justify-content: center !important;
}
"""

with gr.Blocks(title="Machine Maintenance Agentic Automation Suite", theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown("""
    # Machine Maintenance Agentic Automation Suite
    """)

    # # SECTION 1
    # gr.Markdown('<div class="section-header"><h1>Data Upload & Authentication</h1></div>')

    with gr.Group():
        gr.Markdown('<div class="important-note">Upload all required files and enter your credentials to begin.</div>')

        with gr.Row():
            with gr.Column(scale=1):
                machine_logs = gr.File(label="Machine Logs CSV", file_types=[".csv"])
            with gr.Column(scale=1):
                error_manual = gr.File(label="Error Reference Manual", file_types=[".pdf"])
            with gr.Column(scale=1):
                vendor_catalog = gr.File(label="Vendor Catalog", file_types=[".csv"])

        with gr.Row():
            with gr.Column(scale=1):
                email = gr.Textbox(label="Your Email ID", placeholder="your.name@company.com")
            with gr.Column(scale=1):
                password = gr.Textbox(label="Password", placeholder="Enter your password")

    # # SECTION 2
    # gr.Markdown('<div class="section-header"><h2>Step 2: Machine Log Analysis (Agent 1)</h2></div>')
    # with gr.Group():
    #     gr.Markdown('<div class="important-note"><strong>Function:</strong> Analyzes machine logs, identifies errors, and maps them to required parts.</div>')

    # # SECTION 3
    # gr.Markdown('<div class="section-header"><h2>Step 3: Purchase Order Generation (Agent 2)</h2></div>')
    # with gr.Group():
    #     gr.Markdown('<div class="important-note"><strong>Prerequisite:</strong> Complete Step 2 before running this agent.</div>')

    # # SECTION 4
    # gr.Markdown('<div class="section-header"><h2>Vendor Communication</h2></div>')

    with gr.Group():
        gr.Markdown("""
        <div class="important-note">
        Enter vendor email addresses in order.
        The system will map emails to vendors based on the order they appear in your purchase orders.
        </div>
        """)

        with gr.Row():
            v1 = gr.Textbox(label="Vendor 1 Email", placeholder="vendor1@company.com") #, info="First vendor's email address"
            v2 = gr.Textbox(label="Vendor 2 Email", placeholder="vendor2@company.com")
            v3 = gr.Textbox(label="Vendor 3 Email", placeholder="vendor3@company.com")

        with gr.Row():
            v4 = gr.Textbox(label="Vendor 4 Email", placeholder="vendor4@company.com")
            v5 = gr.Textbox(label="Vendor 5 Email", placeholder="vendor5@company.com")

    # # SECTION 5
    # gr.Markdown('<div class="section-header"><h2>Plant Head details</h2></div>')

    with gr.Group():
        gr.Markdown("""
        <div class="important-note">
        A summary report wil be generated.
        This report will be sent to the Plant Head for review.
        </div>
        """)

        plant_head_email = gr.Textbox(
            label="Plant Head Email",
            placeholder="plant.head@company.com",
            info="Email address for report delivery"
        )

    # # SINGLE BUTTON + SINGLE STATUS BOX
    # gr.Markdown('<div class="section-header"><h2>Run Full Workflow</h2></div>')

    with gr.Group():
        with gr.Row(elem_classes=["center-row"]):
            with gr.Column(scale=0):
                run_button = gr.Button("Run Agentic Workflow", variant="primary", size="lg")

        status_box = gr.Textbox(
            label="Agent execution Status (Step-by-step)",
            lines=18,
            interactive=False
        )

        run_button.click(
            run_full_pipeline,
            inputs=[machine_logs, error_manual, vendor_catalog, email, password, v1, v2, v3, v4, v5, plant_head_email],
            outputs=status_box
        )

demo.queue()
demo.launch(debug=True)

/tmp/ipython-input-675757232.py:912: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Machine Maintenance Agentic Automation Suite", theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipython-input-675757232.py:912: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Machine Maintenance Agentic Automation Suite", theme=gr.themes.Soft(), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://15cd454ab147a3384f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[Agent 1] Parsing error manual...
[Agent 1] Parsed 10 error codes from manual.
[Agent 1] Loading logs...
[Agent 1] Matching each log error with manual (severity + parts)...
[Agent 1] Completed. Output saved to: intermediate/agent1_output.csv
[Agent 1] Rows: 190
[Agent 2] Loading vendor catalog...
[Agent 2] Loading Agent 1 output...
[Agent 2] Matching all parts from Agent 1 with vendors...
[Agent 2] WARNING: Some parts had no matching vendor:
   ['Unknown']
[Agent 2] Found 150 error-part-vendor records.
[Agent 2] Creating purchase orders per vendor...
  - Purchase order created for VendorA: invoices/invoice_VendorA.csv
  - Purchase order created for VendorB: invoices/invoice_VendorB.csv
  - Purchase order created for VendorC: invoices/invoice_VendorC.csv
  - Purchase order created for VendorD: invoices/invoice_VendorD.csv
  - Purchase order created for VendorE: invoices/invoice_VendorE.csv
[Agent 3] Sending purchase order emails...
  - Email sent to VendorA (projectbot@eduvance.ai).
  -